# NB05 — Banding Artifact Detection

**Physics rationale:** Dark banding stripes from off-resonance overlap the myocardium at the wrong
frequency offset. The optimal offset places bands *outside* the heart.

**Metrics:** `banded_fraction_myo`, `band_trough_depth`, `banding_burden`, `band_pixel_fraction`

**Selector:** `argmin(banded_fraction_myo)` vs GT ← strongest signal from Exp10 (38.5% Top-1)

**Runs on:** Local CPU — reuses existing `banding.py` and `quality.py`

## 0. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'openpyxl', 'scipy'])
print('Dependencies OK')

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

DATA_ROOT  = r'E:/CINE-Test1/Dr-Dataset/1st_cohort_SAX'
GT_PATH    = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
CACHE_PATH = 'outputs/seg_masks_cache.pkl'
OUT_DIR    = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
EXCLUDE    = ['Patient-KP']
OFFSETS_13 = [-150 + i*25 for i in range(13)]
print('Config OK')

In [ ]:
from utils.dicom_loader import load_all_patients
from utils.gt_loader import load_gt
from utils.seg_wrapper import load_seg_cache

all_data  = load_all_patients(DATA_ROOT, verbose=False)
gt_dict   = load_gt(GT_PATH)
seg_cache = load_seg_cache(CACHE_PATH)
print(f'Loaded {len(all_data)} patients')

## 1. Compute Banding Metrics (via existing freqscout tools)

In [ ]:
from freqscout_agent.tools.banding import BandingDetector
from freqscout_agent.tools.quality import QualityMetrics

detector = BandingDetector(dark_fraction=0.35, min_band_area=40, myo_overlap_threshold=0.05)
quality  = QualityMetrics(background_percentile=5.0)

records = []
band_mask_cache = {}

for pid in sorted(all_data):
    if pid in EXCLUDE: continue
    data = all_data[pid]
    seg  = seg_cache.get(pid)
    if seg is None: continue

    band_results = detector.run(data['images'], data['offsets_hz'], seg)
    qual_results = quality.run(data['images'], data['offsets_hz'], seg)
    band_mask_cache[pid] = [r.band_mask for r in band_results]

    for br, qr in zip(band_results, qual_results):
        records.append({
            'patient_id':          pid,
            'image_no':            br.slice_idx + 1,
            'offset_hz':           br.offset_hz,
            'band_present':        int(br.band_present),
            'bands_in_myo':        int(br.bands_in_myocardium),
            'myo_overlap_frac':    round(br.myo_overlap_fraction, 5),
            'band_depth_in_myo':   round(br.band_depth_in_myo, 5),
            'banding_burden':      round(br.banding_burden, 5),
            'band_pixel_frac':     round(br.band_pixel_fraction, 5),
            'banded_fraction_myo': round(qr.banded_fraction_myo, 5),
            'band_trough_depth':   round(qr.band_trough_depth, 5),
            'severity':            round(br.severity, 5),
            'band_sectors':        ','.join(br.band_sectors),
        })
    print(f'  {pid}: done')

df = pd.DataFrame(records)
df.to_csv(f'{OUT_DIR}/banding_metrics.csv', index=False)
print(f'Computed {len(df)} rows. Saved banding_metrics.csv')
df.head(13)

## 2. Banding Metric Curves

In [ ]:
from utils.vis_utils import plot_metric_curves
import matplotlib.pyplot as plt

for metric_col, title in [
    ('banded_fraction_myo', 'Banded Fraction of Myocardium (Exp10 strongest signal)'),
    ('banding_burden',      'Banding Burden (0=none, 1=severe)'),
    ('band_trough_depth',   'Band Trough Depth in Myocardium'),
]:
    metric_dict = {
        pid: df[df['patient_id']==pid].sort_values('image_no')[metric_col].tolist()
        for pid in sorted(df['patient_id'].unique())
    }
    fig = plot_metric_curves(
        metric_dict=metric_dict, offsets_hz=OFFSETS_13, gt_dict=gt_dict, title=title,
        save_path=f'{OUT_DIR}/banding_{metric_col}_curves.png',
    )
    plt.show()

## 3. Band Mask Overlays — 3 Sample Patients

In [ ]:
import matplotlib.pyplot as plt
from utils.vis_utils import plot_band_overlay

DEMO_PATIENTS = ['Patient-BEV', 'Patient-INA', 'Patient-DEM']

for pid in DEMO_PATIENTS:
    if pid not in band_mask_cache: continue
    masks   = band_mask_cache[pid]
    images  = all_data[pid]['images']
    offsets = all_data[pid]['offsets_hz']
    gt_no   = gt_dict[pid]['image_no']
    n = len(images)

    fig, axes = plt.subplots(1, n, figsize=(n * 2.0, 2.3))
    fig.patch.set_facecolor('#0f0f1a')
    for i, (img, mask, hz) in enumerate(zip(images, masks, offsets)):
        plot_band_overlay(img, mask, ax=axes[i])
        color = 'lime' if (i+1) == gt_no else 'white'
        bf_val = df[(df.patient_id==pid) & (df.image_no==(i+1))]['banded_fraction_myo'].values
        bf_str = f'{bf_val[0]:.2f}' if len(bf_val) else '?'
        axes[i].set_title(f'{hz:+.0f}Hz\nbfm={bf_str}', fontsize=6.5, color=color,
                          fontweight='bold' if color=='lime' else 'normal')

    fig.suptitle(f'{pid} — Band masks (red) | GT = green | bfm = banded_fraction_myo',
                 fontsize=8, color='white')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/banding_overlay_{pid}.png', bbox_inches='tight', dpi=120)
    plt.show()

## 4. Banding Heatmap & Accuracy Evaluation

In [ ]:
from utils.vis_utils import plot_heatmap
from utils.gt_loader import evaluate_predictions
from utils.vis_utils import summary_text
import matplotlib.pyplot as plt

patient_ids_sorted = sorted(df['patient_id'].unique())
n = len(patient_ids_sorted)
matrix = np.zeros((n, 13))
for row_i, pid in enumerate(patient_ids_sorted):
    sub = df[df['patient_id']==pid].sort_values('image_no')
    vals = sub['banded_fraction_myo'].values
    matrix[row_i, :len(vals)] = vals

fig = plot_heatmap(
    data_matrix=matrix, patient_ids=patient_ids_sorted, offsets_hz=OFFSETS_13,
    gt_dict=gt_dict, title='Banded Fraction of Myocardium — lower = better (GT box = green)',
    cbar_label='banded_fraction_myo', cmap='hot_r',
    save_path=f'{OUT_DIR}/banding_bfm_heatmap.png',
)
plt.show()

print('--- Accuracy Evaluation ---')
for selector_col, selector_name, argfn in [
    ('banded_fraction_myo', 'argmin(banded_fraction_myo)', np.argmin),
    ('banding_burden',      'argmin(banding_burden)',      np.argmin),
    ('band_trough_depth',   'argmin(band_trough_depth)',   np.argmin),
]:
    pred_dict = {}
    for pid in df['patient_id'].unique():
        sub = df[df['patient_id']==pid].sort_values('image_no')
        pred_dict[pid] = int(argfn(sub[selector_col].values)) + 1

    res = evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)
    print(summary_text(selector_name, res['top1'], res['pm1'], res['mae_hz'], res['n_valid']))

print('Reference (Exp10): argmin(bfm) -> Top-1 38.5%, +/-1 73.1%, MAE 33.7 Hz')
print('NB05 complete. Proceed to nb06_edge_sharpness.ipynb')